# SafeLens NLA 真实模型可视化展示

这个 notebook 直接加载官方 `Qwen/Qwen2.5-7B-Instruct` 权重，并用官方 NLA 的 AV / AR checkpoint 解释真实 `layer_20.resid_post` activation。

这版示例使用一段短文本：巴黎、法国、埃菲尔铁塔、卢浮宫和最后的问答。主展示不手动挑高语义 token，而是把一句较长完整句子的所有 token 都送进 NLA。

默认整句是：`The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city.`

读图时先看三件事：
- 每个 token 都是这句话里真实 tokenizer 切出来的位置；
- `cosine` / `mse_nrm` 是否说明解释可信；
- NLA explanation 是否真的描述了这个 token 附近的上下文。


## 0. 环境准备

这个 cell 会优先使用当前仓库的 `src/`。如果缺少必要依赖，会从本地 checkout 做 editable install。

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

SAFELENS_INSTALL_EXTRA = "models"
FORCE_EDITABLE_INSTALL = False

def _find_safelens_checkout(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "SafeLens").is_dir():
            return candidate
    return None

def _module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None

PROJECT_ROOT = _find_safelens_checkout(Path.cwd().resolve())
if PROJECT_ROOT is not None:
    src_dir = PROJECT_ROOT / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

core_modules = ["pydantic", "yaml"]
extra_modules = {
    "models": ["torch", "transformers", "safetensors", "huggingface_hub"],
    "viz": ["circuitsvis"],
}
requested_extras = [extra.strip() for extra in SAFELENS_INSTALL_EXTRA.split(",") if extra.strip()]
missing_modules = [
    name
    for name in [
        "SafeLens",
        *core_modules,
        *[m for extra in requested_extras for m in extra_modules[extra]],
    ]
    if _module_missing(name)
]

if FORCE_EDITABLE_INSTALL or missing_modules:
    if PROJECT_ROOT is None:
        raise RuntimeError("SafeLens is not importable and no local checkout was found.")
    install_target = str(PROJECT_ROOT)
    if requested_extras:
        install_target = f"{install_target}[{','.join(requested_extras)}]"
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        install_target,
        "--no-build-isolation",
    ]
    print("Installing SafeLens because these modules are missing:", missing_modules)
    print(" ".join(cmd))
    subprocess.check_call(cmd)
    importlib.invalidate_caches()
else:
    print("SafeLens import path and requested dependencies are ready.")

print("SafeLens project root:", PROJECT_ROOT or "not found")
print("Requested extras:", requested_extras)
print("Missing modules:", missing_modules)


SafeLens import path and requested dependencies are ready.
SafeLens project root: /workspace/SafeLens
Requested extras: ['models']
Missing modules: []


## 1. 加载真实模型、真实 NLA 权重和一段可读文本

这个 cell 会加载 `Qwen/Qwen2.5-7B-Instruct`，然后只缓存官方 NLA 支持的 `layer_20.resid_post`。

示例文本是一个小型阅读理解片段。我们不依赖模型生成长回复，而是直接对这段真实输入跑 forward cache。这样每个被解释 token 都能在原文中找到清楚的上下文。


In [2]:
from pathlib import Path

import torch
from IPython.display import Markdown, display

from SafeLens import (
    ModelLoadConfig,
    NLAClient,
    get_nla_profile,
    plot_nla_fidelity_heatmap,
    plot_nla_result_browser,
)
from SafeLens.utils import build_model_wrapper

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
PROFILE = get_nla_profile("qwen2.5-7b-l20")
CACHE_DIR = Path("../.cache/safelens/nla-qwen2.5-7b-l20").resolve()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "bfloat16" if DEVICE == "cuda" else "float32"
TEXT = (
    "Article: Paris is the capital of France. "
    "The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city. "
    "Question: Which city is the Eiffel Tower in? Answer: Paris."
)

qwen_config = ModelLoadConfig(
    source="transformer_lens",
    name=MODEL_ID,
    dtype=DTYPE,
    device=DEVICE,
    cache_dir=str(CACHE_DIR),
    trust_remote_code=True,
    load_kwargs={"attn_implementation": "eager", "low_cpu_mem_usage": False},
)
qwen_wrapper = build_model_wrapper(qwen_config)
qwen_wrapper.load_model()

token_tensor = qwen_wrapper.to_tokens(TEXT, prepend_bos=True)
tokens = qwen_wrapper.to_str_tokens(token_tensor)
cache_layer = f"layer_{PROFILE.layer}.resid_post"
_, qwen_cache = qwen_wrapper.run_with_cache(
    token_tensor,
    layers=(cache_layer,),
    return_cache_object=False,
)

nla_client = NLAClient.from_profile(
    PROFILE,
    load_reconstructor=True,
    cache_dir=str(CACHE_DIR),
    device=DEVICE,
    dtype=DTYPE,
    local_files_only=False,
)

print("model:", MODEL_ID)
print("device:", DEVICE)
print("token count:", token_tensor.shape[1])
print("cache layer:", cache_layer)
print("cache shape:", tuple(qwen_cache[cache_layer].shape))
display(Markdown("**Text sent through the real model**"))
display(Markdown(TEXT))
print("Tokenized text:")
for idx, token in enumerate(tokens):
    print(f"{idx:02d}: {token!r}")


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|                                  | 0/339 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 339/339 [00:00<00:00, 18275.01it/s]

Fetching 18 files:   0%|                                 | 0/18 [00:00<?, ?it/s]

...

[transformers] Qwen2ForCausalLM LOAD REPORT from: /workspace/.cache/safelens/nla-qwen2.5-7b-l20/models--kitft--nla-qwen2.5-7b-L20-ar/snapshots/e2c9e57eac213d37a31612087f645ab6332c1bb6
Key               | Status  | 
------------------+---------+-
model.norm.weight | MISSING | 
lm_head.weight    | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model: Qwen/Qwen2.5-7B-Instruct
device: cuda
token count: 46
cache layer: layer_20.resid_post
cache shape: (1, 46, 3584)


**Text sent through the real model**

Article: Paris is the capital of France. The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city. Question: Which city is the Eiffel Tower in? Answer: Paris.

Tokenized text:
00: 'Article'
01: ':'
02: ' Paris'
03: ' is'
04: ' the'
05: ' capital'
06: ' of'
07: ' France'
08: '.'
09: ' The'
10: ' E'
11: 'iff'
12: 'el'
13: ' Tower'
14: ' stands'
15: ' in'
16: ' Paris'
17: ','
18: ' and'
19: ' the'
20: ' Lou'
21: 'vre'
22: ' is'
23: ' a'
24: ' museum'
25: ' in'
26: ' the'
27: ' same'
28: ' city'
29: '.'
30: ' Question'
31: ':'
32: ' Which'
33: ' city'
34: ' is'
35: ' the'
36: ' E'
37: 'iff'
38: 'el'
39: ' Tower'
40: ' in'
41: '?'
42: ' Answer'
43: ':'
44: ' Paris'
45: '.'


## 2. 较长整句 token 全量解释

这里不再手动挑 `cosine` 高或语义明显的 token。我们选定一句较长完整句子，然后把这句话的每个 token 都送进官方 NLA：

`The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city.`

对应 tokenizer 位置是 `9..29`，包括 `The`、`Eiffel` 的子词、`Tower`、`Paris`、`Louvre` 的子词、`museum`、`same city` 和句号。

这会让你看到一个更真实的现象：内容词通常更容易解释，功能词、子词和标点也能点击，但它们的解释往往更依赖上下文，也更需要看 `cosine` / `mse_nrm` 判断可信度。


In [3]:
TARGET_SENTENCE = 'The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city.'
TARGET_SENTENCE_POSITIONS = list(range(9, 30))

CONTENT_TOKEN_POSITIONS = {10, 11, 12, 13, 14, 16, 20, 21, 24, 28}
FUNCTION_TOKEN_POSITIONS = {9, 15, 18, 19, 22, 23, 25, 26, 27}
PUNCTUATION_TOKEN_POSITIONS = {17, 29}


def _context_for_position(token_list: list[str], position: int, radius: int = 7) -> str:
    start = max(0, position - radius)
    end = min(len(token_list), position + radius + 1)
    pieces = []
    for idx in range(start, end):
        token = token_list[idx]
        pieces.append(f"[{idx}:{token}]" if idx == position else token)
    return "".join(pieces)


def _token_role(position: int) -> str:
    if position in CONTENT_TOKEN_POSITIONS:
        return "content-or-subword-token"
    if position in FUNCTION_TOKEN_POSITIONS:
        return "function-token"
    if position in PUNCTUATION_TOKEN_POSITIONS:
        return "punctuation"
    return "sentence-token"


def _selection_note(position: int, token: str) -> str:
    role = _token_role(position)
    return (
        f"long-sentence mode: token {position} {token!r} is included because it "
        f"belongs to the target sentence, not because its cosine score was preselected; "
        f"role={role}."
    )


positions = [position for position in TARGET_SENTENCE_POSITIONS if position < len(tokens)]
metadata_by_position = {
    position: {
        "context": _context_for_position(tokens, position),
        "why_selected": _selection_note(position, tokens[position]),
        "role": _token_role(position),
        "sequence": TARGET_SENTENCE,
    }
    for position in positions
}

nla_rows = nla_client.explain_cache(
    qwen_cache,
    tokens=tokens,
    positions=positions,
    sample_id="sentence-eiffel-louvre-all-tokens",
    metadata_by_position=metadata_by_position,
    max_new_tokens=96,
    do_sample=False,
)

print("target sentence:", TARGET_SENTENCE)
print("selected positions:", positions)
print("rows:", len(nla_rows))
for row in nla_rows:
    payload = row.to_dict()
    print(
        payload["token_index"],
        repr(payload["token"]),
        "role=", payload["metadata"].get("role"),
        "cosine=", payload["cosine"],
        "mse_nrm=", payload["mse_nrm"],
    )
    print("context:", payload["metadata"].get("context"))
    print("explanation:", payload["explanation"][:320].replace("\n", " "))
    print("---")


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


target sentence: The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city.
selected positions: [9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
rows: 21
9 ' The' role= function-token cosine= 0.8356015682220459 mse_nrm= 0.3287971317768097
context:  Paris is the capital of France.[9: The] Eiffel Tower stands in Paris
explanation: Chinese language learning format with simple sentences about a city, suggesting a dictionary or grammar book structure with a question about London.  The phrase "London is a big city in England. The" begins a sentence introducing a fact about Paris, implying a list or description of Paris's characteristics or famous la
---
10 ' E' role= content-or-subword-token cosine= 0.8928428292274475 mse_nrm= 0.2143145054578781
context:  is the capital of France. The[10: E]iffel Tower stands in Paris,
explanation: Short English language format with "Hello" greeting structure suggests a simple informational or educationa

## 3. NLA 交互 browser 怎么看

这一块是主视图。现在 token strip 里的 token 覆盖较长完整句子：

`The Eiffel Tower stands in Paris, and the Louvre is a museum in the same city.`

推荐按这个顺序读：

1. 先沿着 token timeline 从左到右点一遍，注意 `Eiffel` 和 `Louvre` 都被 tokenizer 拆成多个子词。
2. 看右侧 `Local context`，确认你点的是原文中的哪一处。
3. 看 `Why this token`，确认它是整句模式自动包含的，不是按 cosine 筛出来的。
4. 看 `cosine` 和 `mse_nrm`：内容词解释通常更稳，功能词、子词和标点要更谨慎。
5. 最后读 `Natural-language explanation`，判断它是否真的概括了这个 token 附近的语义。

注意：NLA 的文字解释不是 ground truth。它是 AV 写出的解释，再由 AR 尝试重构 activation。指标好说明这段文字保留了较多 activation 方向信息，不代表解释唯一正确。


In [4]:
plot_nla_result_browser(
    nla_rows,
    title="Real Qwen2.5 NLA Browser: Longer Sentence Tokens",
)

Visualization(html='<style>.safelens-viz{font-family:Inter,ui-sans-serif,system-ui,sans-serif;line-height:1.35;color:#111827;background:#ffffff;border:1px solid #e5e7eb;border-radius:6px;padding:0.8rem;margin:0.75rem 0;box-sizing:border-box;}.safelens-viz *{box-sizing:border-box;}.safelens-viz h3{font-size:1rem;font-weight:650;margin:0 0 0.65rem 0;color:#0f172a;}.safelens-controls{display:flex;flex-wrap:wrap;gap:0.45rem;margin:0.4rem 0 0.65rem 0;}.safelens-control-label{display:inline-flex;align-items:center;gap:0.35rem;font-size:0.78rem;color:#475569;}.safelens-control-button{border:1px solid #d9dee8;background:#ffffff;color:#111827;border-radius:5px;padding:0.3rem 0.5rem;font-size:0.78rem;cursor:pointer;}.safelens-control-button:hover,.safelens-control-button.is-active{background:#0f172a;color:#ffffff;border-color:#0f172a;}.safelens-select,.safelens-input{border:1px solid #d9dee8;border-radius:5px;padding:0.3rem 0.42rem;font-size:0.8rem;background:#ffffff;color:#111827;}.safelens-hidden{display:none!important;}.safelens-token-strip{display:flex;flex-wrap:wrap;gap:0.08rem;margin:0.35rem 0;}.safelens-token{padding:0.15rem 0.22rem;margin:0.08rem;border-radius:4px;display:inline-block;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-live-token{cursor:pointer;transition:background 120ms ease,color 120ms ease,box-shadow 120ms ease;}.safelens-live-token:hover,.safelens-live-token.is-active{box-shadow:0 0 0 2px rgba(15,23,42,0.22);position:relative;z-index:1;}.safelens-heatmap-scroller{overflow:auto;background:#ffffff;border:1px solid #edf0f5;border-radius:6px;padding:0.35rem;max-width:100%;width:max-content;margin:0 auto;}.safelens-heatmap-svg{display:block;background:#ffffff;width:auto;height:auto;max-width:none;margin:0 auto;}.safelens-heatmap-background{fill:#ffffff;}.safelens-heatmap-label{font-size:11px;fill:#334155;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-axis-label{font-size:12px;fill:#475569;font-weight:600;}.safelens-heatmap-value{font-size:10px;font-weight:650;pointer-events:none;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-heatmap-cell{cursor:crosshair;stroke:#ffffff;stroke-width:1;transition:stroke 80ms ease,stroke-width 80ms ease;}.safelens-heatmap-cell.safelens-row-focus,.safelens-heatmap-cell.safelens-col-focus{stroke:#64748b;stroke-width:1.6;}.safelens-heatmap-cell.safelens-cell-focus{stroke:#0f172a;stroke-width:2.4;}.safelens-axis{font-size:0.78rem;color:#4b5563;margin:0.25rem 0;}.safelens-x-axis{margin-left:2.4rem;}.safelens-focus-readout{min-height:1rem;font-size:0.78rem;color:#374151;margin:0.3rem 0;}.safelens-muted{font-size:0.78rem;color:#64748b;padding:0.35rem 0;}.safelens-rank-list{display:grid;gap:0.42rem;margin-top:0.55rem;}.safelens-rank-item{border:1px solid #e5e7eb;background:#ffffff;border-radius:6px;padding:0.48rem 0.55rem;}.safelens-rank-main{display:flex;justify-content:space-between;align-items:baseline;gap:0.75rem;min-width:0;}.safelens-rank-label{font-size:0.8rem;font-weight:650;color:#111827;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;}.safelens-rank-value{font-size:0.78rem;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-weight:700;white-space:nowrap;}.safelens-rank-value.is-positive{color:#be123c;}.safelens-rank-value.is-negative{color:#1d4ed8;}.safelens-rank-track{height:0.42rem;background:#f1f5f9;border-radius:999px;overflow:hidden;margin-top:0.38rem;}.safelens-rank-fill{display:block;height:100%;border-radius:999px;}.safelens-rank-fill.is-positive{background:#e11d48;}.safelens-rank-fill.is-negative{background:#2563eb;}.safelens-direction-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:0.7rem;margin-top:0.55rem;align-items:start;}.safelens-direction-panel{border:1px solid #e5e7eb;background:#ffffff;border-radius:6px;padding:0.55rem;min-width:0;}.safelens-panel-title{font-size:0.82rem;font-weight:700;color:#0f172a;margin:0 0 0.5rem 0;}.safelens-nla-browser{max

## 4. Fidelity heatmap 怎么看

heatmap 展示同一句较长句子里每个 token 的 fidelity。它不是“挑出来的几个好 token”，而是一句完整 token 序列。

- `cosine` 图：颜色越深，说明 AR 从 NLA 解释重构出的方向越接近真实 activation。
- `mse_nrm` 图：数值越低越好。它能帮你发现哪些 token 的 explanation 不该过度解读。
- 子词、功能词和标点如果指标差，这是正常诊断信息，不是 bug。


In [5]:
plot_nla_fidelity_heatmap(
    nla_rows,
    metric="cosine",
    title="Real NLA Cosine Fidelity: Longer Sentence Tokens",
)

Visualization(html='<style>.safelens-viz{font-family:Inter,ui-sans-serif,system-ui,sans-serif;line-height:1.35;color:#111827;background:#ffffff;border:1px solid #e5e7eb;border-radius:6px;padding:0.8rem;margin:0.75rem 0;box-sizing:border-box;}.safelens-viz *{box-sizing:border-box;}.safelens-viz h3{font-size:1rem;font-weight:650;margin:0 0 0.65rem 0;color:#0f172a;}.safelens-controls{display:flex;flex-wrap:wrap;gap:0.45rem;margin:0.4rem 0 0.65rem 0;}.safelens-control-label{display:inline-flex;align-items:center;gap:0.35rem;font-size:0.78rem;color:#475569;}.safelens-control-button{border:1px solid #d9dee8;background:#ffffff;color:#111827;border-radius:5px;padding:0.3rem 0.5rem;font-size:0.78rem;cursor:pointer;}.safelens-control-button:hover,.safelens-control-button.is-active{background:#0f172a;color:#ffffff;border-color:#0f172a;}.safelens-select,.safelens-input{border:1px solid #d9dee8;border-radius:5px;padding:0.3rem 0.42rem;font-size:0.8rem;background:#ffffff;color:#111827;}.safelens-hidden{display:none!important;}.safelens-token-strip{display:flex;flex-wrap:wrap;gap:0.08rem;margin:0.35rem 0;}.safelens-token{padding:0.15rem 0.22rem;margin:0.08rem;border-radius:4px;display:inline-block;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-live-token{cursor:pointer;transition:background 120ms ease,color 120ms ease,box-shadow 120ms ease;}.safelens-live-token:hover,.safelens-live-token.is-active{box-shadow:0 0 0 2px rgba(15,23,42,0.22);position:relative;z-index:1;}.safelens-heatmap-scroller{overflow:auto;background:#ffffff;border:1px solid #edf0f5;border-radius:6px;padding:0.35rem;max-width:100%;width:max-content;margin:0 auto;}.safelens-heatmap-svg{display:block;background:#ffffff;width:auto;height:auto;max-width:none;margin:0 auto;}.safelens-heatmap-background{fill:#ffffff;}.safelens-heatmap-label{font-size:11px;fill:#334155;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-axis-label{font-size:12px;fill:#475569;font-weight:600;}.safelens-heatmap-value{font-size:10px;font-weight:650;pointer-events:none;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-heatmap-cell{cursor:crosshair;stroke:#ffffff;stroke-width:1;transition:stroke 80ms ease,stroke-width 80ms ease;}.safelens-heatmap-cell.safelens-row-focus,.safelens-heatmap-cell.safelens-col-focus{stroke:#64748b;stroke-width:1.6;}.safelens-heatmap-cell.safelens-cell-focus{stroke:#0f172a;stroke-width:2.4;}.safelens-axis{font-size:0.78rem;color:#4b5563;margin:0.25rem 0;}.safelens-x-axis{margin-left:2.4rem;}.safelens-focus-readout{min-height:1rem;font-size:0.78rem;color:#374151;margin:0.3rem 0;}.safelens-muted{font-size:0.78rem;color:#64748b;padding:0.35rem 0;}.safelens-rank-list{display:grid;gap:0.42rem;margin-top:0.55rem;}.safelens-rank-item{border:1px solid #e5e7eb;background:#ffffff;border-radius:6px;padding:0.48rem 0.55rem;}.safelens-rank-main{display:flex;justify-content:space-between;align-items:baseline;gap:0.75rem;min-width:0;}.safelens-rank-label{font-size:0.8rem;font-weight:650;color:#111827;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;}.safelens-rank-value{font-size:0.78rem;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-weight:700;white-space:nowrap;}.safelens-rank-value.is-positive{color:#be123c;}.safelens-rank-value.is-negative{color:#1d4ed8;}.safelens-rank-track{height:0.42rem;background:#f1f5f9;border-radius:999px;overflow:hidden;margin-top:0.38rem;}.safelens-rank-fill{display:block;height:100%;border-radius:999px;}.safelens-rank-fill.is-positive{background:#e11d48;}.safelens-rank-fill.is-negative{background:#2563eb;}.safelens-direction-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:0.7rem;margin-top:0.55rem;align-items:start;}.safelens-direction-panel{border:1px solid #e5e7eb;background:#ffffff;border-radius:6px;padding:0.55rem;min-width:0;}.safelens-panel-title{font-size:0.82rem;font-weight:700;color:#0f172a;margin:0 0 0.5rem 0;}.safelens-nla-browser{max

In [6]:
plot_nla_fidelity_heatmap(
    nla_rows,
    metric="mse_nrm",
    title="Real NLA Reconstruction Error: Longer Sentence Tokens",
)

Visualization(html='<style>.safelens-viz{font-family:Inter,ui-sans-serif,system-ui,sans-serif;line-height:1.35;color:#111827;background:#ffffff;border:1px solid #e5e7eb;border-radius:6px;padding:0.8rem;margin:0.75rem 0;box-sizing:border-box;}.safelens-viz *{box-sizing:border-box;}.safelens-viz h3{font-size:1rem;font-weight:650;margin:0 0 0.65rem 0;color:#0f172a;}.safelens-controls{display:flex;flex-wrap:wrap;gap:0.45rem;margin:0.4rem 0 0.65rem 0;}.safelens-control-label{display:inline-flex;align-items:center;gap:0.35rem;font-size:0.78rem;color:#475569;}.safelens-control-button{border:1px solid #d9dee8;background:#ffffff;color:#111827;border-radius:5px;padding:0.3rem 0.5rem;font-size:0.78rem;cursor:pointer;}.safelens-control-button:hover,.safelens-control-button.is-active{background:#0f172a;color:#ffffff;border-color:#0f172a;}.safelens-select,.safelens-input{border:1px solid #d9dee8;border-radius:5px;padding:0.3rem 0.42rem;font-size:0.8rem;background:#ffffff;color:#111827;}.safelens-hidden{display:none!important;}.safelens-token-strip{display:flex;flex-wrap:wrap;gap:0.08rem;margin:0.35rem 0;}.safelens-token{padding:0.15rem 0.22rem;margin:0.08rem;border-radius:4px;display:inline-block;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-live-token{cursor:pointer;transition:background 120ms ease,color 120ms ease,box-shadow 120ms ease;}.safelens-live-token:hover,.safelens-live-token.is-active{box-shadow:0 0 0 2px rgba(15,23,42,0.22);position:relative;z-index:1;}.safelens-heatmap-scroller{overflow:auto;background:#ffffff;border:1px solid #edf0f5;border-radius:6px;padding:0.35rem;max-width:100%;width:max-content;margin:0 auto;}.safelens-heatmap-svg{display:block;background:#ffffff;width:auto;height:auto;max-width:none;margin:0 auto;}.safelens-heatmap-background{fill:#ffffff;}.safelens-heatmap-label{font-size:11px;fill:#334155;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-axis-label{font-size:12px;fill:#475569;font-weight:600;}.safelens-heatmap-value{font-size:10px;font-weight:650;pointer-events:none;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;}.safelens-heatmap-cell{cursor:crosshair;stroke:#ffffff;stroke-width:1;transition:stroke 80ms ease,stroke-width 80ms ease;}.safelens-heatmap-cell.safelens-row-focus,.safelens-heatmap-cell.safelens-col-focus{stroke:#64748b;stroke-width:1.6;}.safelens-heatmap-cell.safelens-cell-focus{stroke:#0f172a;stroke-width:2.4;}.safelens-axis{font-size:0.78rem;color:#4b5563;margin:0.25rem 0;}.safelens-x-axis{margin-left:2.4rem;}.safelens-focus-readout{min-height:1rem;font-size:0.78rem;color:#374151;margin:0.3rem 0;}.safelens-muted{font-size:0.78rem;color:#64748b;padding:0.35rem 0;}.safelens-rank-list{display:grid;gap:0.42rem;margin-top:0.55rem;}.safelens-rank-item{border:1px solid #e5e7eb;background:#ffffff;border-radius:6px;padding:0.48rem 0.55rem;}.safelens-rank-main{display:flex;justify-content:space-between;align-items:baseline;gap:0.75rem;min-width:0;}.safelens-rank-label{font-size:0.8rem;font-weight:650;color:#111827;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;}.safelens-rank-value{font-size:0.78rem;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-weight:700;white-space:nowrap;}.safelens-rank-value.is-positive{color:#be123c;}.safelens-rank-value.is-negative{color:#1d4ed8;}.safelens-rank-track{height:0.42rem;background:#f1f5f9;border-radius:999px;overflow:hidden;margin-top:0.38rem;}.safelens-rank-fill{display:block;height:100%;border-radius:999px;}.safelens-rank-fill.is-positive{background:#e11d48;}.safelens-rank-fill.is-negative{background:#2563eb;}.safelens-direction-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:0.7rem;margin-top:0.55rem;align-items:start;}.safelens-direction-panel{border:1px solid #e5e7eb;background:#ffffff;border-radius:6px;padding:0.55rem;min-width:0;}.safelens-panel-title{font-size:0.82rem;font-weight:700;color:#0f172a;margin:0 0 0.5rem 0;}.safelens-nla-browser{max

## 5. 读图时最容易误解的点

- 现在 token 是较长整句全量选择，不是按高 `cosine` 预筛选。
- `E`、`iff`、`el` 共同组成 `Eiffel`，`Lou`、`vre` 共同组成 `Louvre`，解释时要结合 `Local context` 看。
- `Tower`、`Paris`、`museum`、`city` 这类内容词更容易得到看起来稳定的解释。
- `The`、`in`、`and`、`the`、`is`、`a`、`.` 这类功能词和标点也能点击，但解释更依赖上下文，必须结合 `cosine` / `mse_nrm` 看。
- `activation_norm` 是原始向量的大小；NLA 解释主要看方向，所以更应该关注 `cosine` 和 `mse_nrm`。
- 当前官方 NLA profile 只覆盖 `Qwen2.5-7B-Instruct` 的 `layer_20.resid_post`，所以这个 notebook 必须加载真实 Qwen2.5 权重，不能拿 Qwen3 或手写数据替代。
